# 07.7 Unicode, Encoding, and `bytes` vs `str`

This is the topic behind most text bugs: mojibake, `UnicodeDecodeError`, emoji
that count as two characters, and files that open fine on one machine and break
on another.

## The one idea

- **`str`** is a sequence of **characters** — what you work with in Python
- **`bytes`** is a sequence of **numbers 0–255** — what files and networks store

An **encoding** converts between them. Get the encoding wrong and you get
garbage or an exception.

```
   str  ---- .encode() ---->  bytes
   str  <--- .decode() -----  bytes
```

## Easy — Characters and their numbers

Every character has a number. That is the whole foundation.

In [ ]:
# EXAMPLE 1: Finding a character's number
# ord() gives the code point - the number assigned to a character.
print(ord("A"))
print(ord("a"))
print(ord("0"))

In [ ]:
# EXAMPLE 2: Going the other way
# chr() turns a number back into a character.
print(chr(65))
print(chr(97))
print(chr(48))

In [ ]:
# EXAMPLE 3: The alphabet is consecutive
# Letters have consecutive code points, which makes arithmetic work.
for letter in "ABC":
    print(letter, ord(letter))

In [ ]:
# EXAMPLE 4: Uppercase and lowercase differ by 32
print("A is", ord("A"))
print("a is", ord("a"))
print("Difference:", ord("a") - ord("A"))

In [ ]:
# EXAMPLE 5: Unicode covers every language
# Python 3 strings hold any character, not just English.
samples = ["hello", "नमस्ते", "こんにちは", "مرحبا", "Привет"]

for text in samples:
    print(f"{text:<12} {len(text)} characters")

In [ ]:
# EXAMPLE 6: Emoji are just characters too
smiley = chr(128512)

print(smiley)
print("Code point:", ord(smiley))
print("Length in Python:", len(smiley))

## Medium — Encoding and decoding

Turning text into bytes for storage, and back again.

In [ ]:
# EXAMPLE 7: Encoding text to bytes
# encode() produces bytes, shown with a b prefix.
text = "hello"
raw = text.encode("utf-8")

print("text:", text, type(text).__name__)
print("raw: ", raw, type(raw).__name__)

In [ ]:
# EXAMPLE 8: Decoding bytes back to text
raw = b"hello"
text = raw.decode("utf-8")

print("raw: ", raw)
print("text:", text)

In [ ]:
# EXAMPLE 9: ASCII characters are one byte each
text = "hello"
raw = text.encode("utf-8")

print("characters:", len(text))
print("bytes:     ", len(raw))
print("the bytes: ", list(raw))

In [ ]:
# EXAMPLE 10: Non-ASCII characters need more bytes
# UTF-8 uses 1 to 4 bytes per character.
samples = ["a", "é", "€", chr(128512)]

for character in samples:
    raw = character.encode("utf-8")
    print(f"{character:<3} code point {ord(character):>7}  "
          f"{len(raw)} byte(s): {list(raw)}")

In [ ]:
# EXAMPLE 11: Length in characters versus bytes
# This difference causes real bugs in field-length validation.
text = "café"

print("text:      ", text)
print("characters:", len(text))
print("utf-8 bytes:", len(text.encode("utf-8")))

In [ ]:
# EXAMPLE 12: Bytes and str cannot be mixed
# Python 3 refuses to combine them, which prevents a whole bug class.
text = "hello"
raw = b"hello"

print("Equal?", text == raw)

try:
    text + raw
except TypeError as error:
    print("Concatenating:", error)

In [ ]:
# EXAMPLE 13: Choosing the wrong encoding produces mojibake
# Encode as UTF-8, decode as Latin-1, and the text is corrupted.
original = "café"

raw = original.encode("utf-8")
wrong = raw.decode("latin-1")

print("original:", original)
print("wrong:   ", wrong)
print("")
print("This garbled text has a name: mojibake.")

In [ ]:
# EXAMPLE 14: An encoding that cannot represent the character
# ASCII has no accented letters, so encoding fails.
text = "café"

try:
    text.encode("ascii")
except UnicodeEncodeError as error:
    print("UnicodeEncodeError:", error)

In [ ]:
# EXAMPLE 15: Handling encoding errors
# The errors argument decides what happens to impossible characters.
text = "café"

print("ignore: ", text.encode("ascii", errors="ignore"))
print("replace:", text.encode("ascii", errors="replace"))
print("xmlcharrefreplace:", text.encode("ascii", errors="xmlcharrefreplace"))
print("backslashreplace: ", text.encode("ascii", errors="backslashreplace"))

In [ ]:
# EXAMPLE 16: Decoding invalid bytes
# Not every byte sequence is valid UTF-8.
invalid = b"\xff\xfe"

try:
    invalid.decode("utf-8")
except UnicodeDecodeError as error:
    print("UnicodeDecodeError:", error)

print("")
print("with errors='replace':", invalid.decode("utf-8", errors="replace"))

In [ ]:
# EXAMPLE 17: Always specify the encoding when opening files
# The default depends on the operating system, which breaks portability.
import locale

print("This machine's preferred encoding:", locale.getpreferredencoding())
print("")
print("RISKY:  open('data.txt')")
print("SAFE:   open('data.txt', encoding='utf-8')")
print("")
print("Always pass encoding explicitly. Chapter 22 covers files.")

## Hard — Normalisation and the hard parts

Where Unicode stops being simple.

In [ ]:
# EXAMPLE 18: The same text, two different encodings
# 'é' can be one character, or 'e' plus a combining accent.
single = "caf\u00e9"
combined = "cafe\u0301"

print("single:  ", single, "length", len(single))
print("combined:", combined, "length", len(combined))
print("Look identical but equal?", single == combined)

In [ ]:
# EXAMPLE 19: Normalisation fixes the comparison
# unicodedata.normalize converts to a canonical form.
import unicodedata

single = "caf\u00e9"
combined = "cafe\u0301"

normalised_single = unicodedata.normalize("NFC", single)
normalised_combined = unicodedata.normalize("NFC", combined)

print("After NFC normalisation, equal?",
      normalised_single == normalised_combined)

In [ ]:
# EXAMPLE 20: The four normalisation forms
import unicodedata

text = "cafe\u0301"

for form in ["NFC", "NFD", "NFKC", "NFKD"]:
    result = unicodedata.normalize(form, text)
    print(f"{form}: length {len(result)}  {result}")

print("")
print("NFC composes (fewer characters). NFD decomposes.")
print("The K forms also convert compatibility characters.")

In [ ]:
# EXAMPLE 21: Normalise before comparing user input
# The practical rule for any text that came from a user.
import unicodedata


def same_text(first, second):
    """Compare two strings the way a human would."""
    # Normalise the encoding, then fold the case.
    return (unicodedata.normalize("NFC", first).casefold()
            == unicodedata.normalize("NFC", second).casefold())


print(same_text("caf\u00e9", "cafe\u0301"))
print(same_text("STRASSE", "strasse"))
print(same_text("hello", "world"))

In [ ]:
# EXAMPLE 22: Emoji can be several code points
# A family emoji is built from several characters joined together.
family = chr(128104) + chr(8205) + chr(128105) + chr(8205) + chr(128102)

print(family)
print("len() reports:", len(family))
print("But a human sees one symbol.")
print("")
print("For user-visible character counts, a library like `regex`")
print("with grapheme cluster support is needed.")

In [ ]:
# EXAMPLE 23: Looking up character information
import unicodedata

for character in ["A", "é", "9", chr(128512)]:
    name = unicodedata.name(character, "UNKNOWN")
    category = unicodedata.category(character)
    print(f"{character:<3} {category}  {name}")

In [ ]:
# EXAMPLE 24: Stripping accents from text
# Decompose, then drop the combining marks.
import unicodedata


def remove_accents(text):
    """Return the text with accents removed."""
    # NFD splits characters into base plus combining marks.
    decomposed = unicodedata.normalize("NFD", text)

    # Category Mn means "mark, non-spacing" - the accents.
    return "".join(character for character in decomposed
                   if unicodedata.category(character) != "Mn")


for text in ["café", "naïve", "Zürich"]:
    print(f"{text:<10} -> {remove_accents(text)}")

In [ ]:
# EXAMPLE 25: Working with bytes directly
# bytes behaves like an immutable sequence of integers.
raw = "hello".encode("utf-8")

print("the bytes:  ", raw)
print("first byte: ", raw[0], "<- an int, not a character")
print("as a list:  ", list(raw))
print("hex:        ", raw.hex())
print("from hex:   ", bytes.fromhex("68656c6c6f"))

In [ ]:
# EXAMPLE 26: bytearray is the mutable version
# Use bytearray when you need to modify binary data in place.
buffer = bytearray(b"hello")

buffer[0] = ord("H")
buffer.append(ord("!"))

print(buffer)
print("as text:", buffer.decode("utf-8"))

## Takeaways

1. **`str` is characters; `bytes` is numbers.** An encoding converts between them.
2. `"text".encode("utf-8")` produces bytes; `b"...".decode("utf-8")` reads them
   back.
3. **UTF-8 uses 1–4 bytes per character**, so `len(text)` and
   `len(text.encode())` differ for non-ASCII text.
4. Python 3 **refuses to mix** `str` and `bytes` — deliberately.
5. Decoding with the wrong encoding gives **mojibake**; encoding an impossible
   character raises `UnicodeEncodeError`.
6. **Always pass `encoding="utf-8"` when opening files.** The default is
   platform-dependent.
7. The same visible text can have **two encodings** (`é` vs `e` + accent) —
   normalise with `unicodedata.normalize("NFC", text)` before comparing.
8. Some emoji are **several code points**, so `len()` does not match what a human
   sees.

## Try it yourself

1. Print the code point of the first letter of your name.
2. Encode a word with an accent and count the bytes versus the characters.
3. Encode as UTF-8, decode as Latin-1, and look at the mojibake.
4. Compare `"caf\u00e9"` with `"cafe\u0301"` before and after normalising.
5. Write a function that strips accents from a name.